In [30]:
# If needed, uncomment:
# %pip install -q nltk gensim matplotlib

import re
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec

print("Libraries imported successfully.")

Libraries imported successfully.


In [32]:
for resource in ["gutenberg", "punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

text = gutenberg.raw("carroll-alice.txt")
print("Dataset loaded.")
print("Total characters:", len(text))
print(text[:500])

Dataset loaded.
Total characters: 144395
[Alice's Adventures in Wonderland by Lewis Carroll 1865]

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy an


In [34]:
def preprocess_text(raw_text):
    sentences = sent_tokenize(raw_text)
    cleaned_sentences = []
    stop_words = set(stopwords.words("english"))

    for sentence in sentences:
        sentence = sentence.lower()
        sentence = re.sub(r"[^a-zA-Z\s]", " ", sentence)
        words = word_tokenize(sentence)
        words = [word for word in words if word not in stop_words and len(word) > 1]
        if words:
            cleaned_sentences.append(words)

    return cleaned_sentences

processed_sentences = preprocess_text(text)
print("Total processed sentences:", len(processed_sentences))
print("Sample processed sentences:", processed_sentences[:3])

Total processed sentences: 1590
Sample processed sentences: [['alice', 'adventures', 'wonderland', 'lewis', 'carroll', 'chapter'], ['rabbit', 'hole', 'alice', 'beginning', 'get', 'tired', 'sitting', 'sister', 'bank', 'nothing', 'twice', 'peeped', 'book', 'sister', 'reading', 'pictures', 'conversations', 'use', 'book', 'thought', 'alice', 'without', 'pictures', 'conversation'], ['considering', 'mind', 'well', 'could', 'hot', 'day', 'made', 'feel', 'sleepy', 'stupid', 'whether', 'pleasure', 'making', 'daisy', 'chain', 'would', 'worth', 'trouble', 'getting', 'picking', 'daisies', 'suddenly', 'white', 'rabbit', 'pink', 'eyes', 'ran', 'close']]


In [36]:
model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,            # CBOW architecture
    workers=4,
    epochs=50,
    seed=42
)
print("CBOW model training complete.")
print("Vocabulary size:", len(model.wv))

CBOW model training complete.
Vocabulary size: 1313


In [38]:
target_words = ["alice", "rabbit", "queen", "cat"]

for word in target_words:
    print(f"\nWords most similar to '{word}':")
    if word in model.wv:
        for similar_word, score in model.wv.most_similar(word, topn=5):
            print(f"  {similar_word}: {score:.4f}")
    else:
        print("  Word is not present in the model vocabulary.")


Words most similar to 'alice':
  politely: 0.9177
  answer: 0.9092
  eggs: 0.9077
  girl: 0.9069
  sharply: 0.9068

Words most similar to 'rabbit':
  kid: 0.9263
  white: 0.9081
  directed: 0.9074
  gloves: 0.8955
  fan: 0.8639

Words most similar to 'queen':
  knave: 0.9454
  stopped: 0.9391
  loud: 0.9384
  top: 0.9358
  timid: 0.9348

Words most similar to 'cat':
  interesting: 0.9333
  talk: 0.9326
  better: 0.9307
  must: 0.9296
  deal: 0.9287


In [40]:
context = ["white", "rabbit"]
try:
    predictions = model.predict_output_word(context, topn=5)
    if predictions:
        print(f"Predictions for context {context}:")
        for predicted_word, probability in predictions:
            print(f"  {predicted_word}: {probability:.4f}")
    else:
        print("No predictions returned for this context.")
except (AttributeError, TypeError, ValueError) as exc:
    print("This Gensim version does not support prediction for this model:", exc)
    print("Use most_similar() above to inspect learned word relationships.")

Predictions for context ['white', 'rabbit']:
  white: 0.6398
  rabbit: 0.2550
  gloves: 0.0094
  three: 0.0053
  kid: 0.0048
